# Statistics for ML Quality: Are These Two Models Actually Different?

A teammate just swapped your team's logistic regression baseline for a random forest on the
breast-cancer malignancy classifier and claims "it's better." Your job, as the QA engineer on
the ML team, is to find out whether that claim survives contact with statistics, not just a
single accuracy number that happens to look bigger.

You'll use the real [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)
dataset (via `sklearn.datasets.load_breast_cancer`) — 569 real patient tumor samples with 30
real-valued diagnostic features, labeled malignant/benign.

**What you should produce:**
1. A `bootstrap_ci(y_true, y_pred, metric_fn, ...)` function that returns a point estimate and a
   95% confidence interval for any classification metric, built from resampling the test set.
2. Bootstrap 95% CIs for accuracy and F1 for both a `LogisticRegression` baseline and a
   `RandomForestClassifier` challenger, trained on the same train/test split.
3. A `mcnemar_test(y_true, pred_a, pred_b)` function implementing McNemar's test (a **paired**
   test on the same test samples — more powerful than comparing two independent CIs when you
   want to know if model B beats model A on the *same* data).
4. A short written verdict: is the random forest's improvement statistically significant, and
   why might overlapping/non-overlapping bootstrap CIs give a misleading answer compared to the
   paired test?
5. **Bonus (QA habit worth building):** a train/test distribution check — run a Kolmogorov-
   Smirnov test per feature between the train and test splits to flag any feature whose
   distribution shifted across the split (a miniature version of the drift checks you'd run
   in production before trusting a model's offline evaluation at all).


## Starter cells

Fill in every `# TODO`. Don't look at the solution until you've tried each function yourself.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42


In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="malignant_is_0")  # sklearn encodes malignant=0, benign=1

print(X.shape)
X.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

# TODO: fit a LogisticRegression baseline (increase max_iter so it converges) on X_train/y_train
log_reg = None

# TODO: fit a RandomForestClassifier challenger (n_estimators=200) on X_train/y_train
rf = None

# TODO: get predictions from both models on X_test
pred_log_reg = None
pred_rf = None


In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, random_state=42):
    """Return (point_estimate, ci_lower, ci_upper, boot_distribution) for metric_fn via
    resampling (y_true, y_pred) pairs together with replacement."""
    # TODO: implement bootstrap resampling of the *paired* (y_true, y_pred) arrays,
    # recompute metric_fn on each resample, and take the percentile interval.
    raise NotImplementedError


In [ ]:
def mcnemar_test(y_true, pred_a, pred_b):
    """Return (statistic, p_value) for McNemar's test comparing two classifiers' errors
    on the SAME test samples. Use the continuity-corrected statistic."""
    # TODO: build the 2x2 contingency table of (model_a correct/incorrect) x (model_b correct/incorrect),
    # then compute the continuity-corrected McNemar chi-square statistic and its p-value
    # (1 degree of freedom).
    raise NotImplementedError


In [ ]:
# TODO: compute bootstrap CIs for accuracy and F1 for both models and print them clearly

# TODO: plot the two models' bootstrap accuracy distributions on one histogram, with their
# point estimates and CI bounds marked, so you can SEE whether the CIs overlap

# TODO: run mcnemar_test(y_test, pred_log_reg, pred_rf) and print the statistic and p-value


In [ ]:
# Bonus: train/test distribution check with the Kolmogorov-Smirnov test
# TODO: for each column in X, run scipy.stats.ks_2samp(X_train[col], X_test[col]) and
# collect the features where p < 0.05. Think about why checking 30 features at a 0.05
# threshold individually is itself a statistical pitfall.


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Fitting the two models:**

```python
log_reg = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
log_reg.fit(X_train, y_train)
pred_log_reg = log_reg.predict(X_test)

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
```

**`bootstrap_ci`** — resample *paired* indices (not `y_true` and `y_pred` independently —
that would break the correspondence between a sample's true label and its prediction):

```python
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    point = metric_fn(y_true, y_pred)
    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_stats[i] = metric_fn(y_true[idx], y_pred[idx])
    lower, upper = np.percentile(boot_stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper, boot_stats
```

**`mcnemar_test`** — this is a *paired* test: it only looks at the samples where the two
models disagree (discordant pairs), which is exactly what makes it more powerful than
comparing two independent confidence intervals for the same comparison:

```python
def mcnemar_test(y_true, pred_a, pred_b):
    y_true = np.asarray(y_true)
    correct_a = (np.asarray(pred_a) == y_true)
    correct_b = (np.asarray(pred_b) == y_true)
    b = np.sum(correct_a & ~correct_b)   # a right, b wrong
    c = np.sum(~correct_a & correct_b)   # a wrong, b right
    if b + c == 0:
        return 0.0, 1.0
    statistic = (abs(b - c) - 1) ** 2 / (b + c)
    p_value = 1 - stats.chi2.cdf(statistic, df=1)
    return statistic, p_value
```

**Running the comparison:**

```python
acc_log_reg = bootstrap_ci(y_test, pred_log_reg, accuracy_score)
acc_rf = bootstrap_ci(y_test, pred_rf, accuracy_score)
f1_log_reg = bootstrap_ci(y_test, pred_log_reg, f1_score)
f1_rf = bootstrap_ci(y_test, pred_rf, f1_score)

print(f"LogReg accuracy: {acc_log_reg[0]:.3f}  95% CI [{acc_log_reg[1]:.3f}, {acc_log_reg[2]:.3f}]")
print(f"RF     accuracy: {acc_rf[0]:.3f}  95% CI [{acc_rf[1]:.3f}, {acc_rf[2]:.3f}]")
print(f"LogReg F1:       {f1_log_reg[0]:.3f}  95% CI [{f1_log_reg[1]:.3f}, {f1_log_reg[2]:.3f}]")
print(f"RF     F1:       {f1_rf[0]:.3f}  95% CI [{f1_rf[1]:.3f}, {f1_rf[2]:.3f}]")

plt.hist(acc_log_reg[3], bins=40, alpha=0.5, label="LogReg bootstrap accuracy")
plt.hist(acc_rf[3], bins=40, alpha=0.5, label="RF bootstrap accuracy")
plt.axvline(acc_log_reg[0], color="C0", linestyle="--")
plt.axvline(acc_rf[0], color="C1", linestyle="--")
plt.legend()
plt.xlabel("accuracy")
plt.title("Bootstrap accuracy distributions")
plt.show()

stat, p = mcnemar_test(y_test, pred_log_reg, pred_rf)
print(f"McNemar statistic={stat:.3f}, p-value={p:.4f}")
```

**Why both tools matter, and why they can disagree:**

The two bootstrap CIs are built independently from each model's *own* resamples, so they
ignore the fact that both models were scored on the *exact same* test samples. Two
independent CIs can overlap even when one model is reliably better on a sample-by-sample
basis — overlap tells you "I can't rule out they're equal from these two intervals alone,"
not "they're equal." McNemar's test fixes this by looking only at the samples where the
two models disagree, which is a strictly more powerful, paired comparison for exactly this
question. The practical QA takeaway: when comparing two models on the *same* test set, a
paired test (McNemar, a paired bootstrap difference, or a paired t-test on per-fold scores)
is the right tool — not two separate confidence intervals eyeballed for overlap. If
`mcnemar_test` gives p < 0.05, you have real evidence the random forest's error pattern
differs from the logistic regression's on this test set; if the bootstrap CIs also barely
overlap or don't overlap at all, that's corroborating (not required) evidence pointing the
same way.

**Bonus — KS-test drift check between train and test:**

```python
drifted = []
for col in X.columns:
    ks_stat, p_value = stats.ks_2samp(X_train[col], X_test[col])
    if p_value < 0.05:
        drifted.append((col, ks_stat, p_value))

print(f"{len(drifted)} / {len(X.columns)} features flagged at p < 0.05:")
for col, ks_stat, p_value in drifted:
    print(f"  {col}: KS={ks_stat:.3f}, p={p_value:.4f}")
```

With a random stratified split this list should be small or empty — and that's the point
of running the check, not the headline. The pitfall to notice: testing 30 features at
p < 0.05 each gives a double-digit percentage chance of at least one false positive purely
from multiple comparisons (roughly `1 - 0.95**30 ≈ 79%`). In a real drift-monitoring
pipeline you'd apply a multiple-testing correction (Bonferroni, Benjamini-Hochberg) or raise
the significance bar before paging anyone about "drift."

</details>